# Beyond Foundational LLMs: Agents, Emergent Abilities, and Advanced Reasoning

## 1. Overview & Prerequisites

This notebook explores the cutting-edge concepts discussed in the Stanford CS25 lecture, moving beyond standard Large Language Models (LLMs) to explore what's next: autonomous agents, the surprising phenomenon of emergent abilities, advanced reasoning techniques, and the push for data efficiency with projects like BabyLM.

### 1.1 Summary of Topics

This notebook synthesizes the lecture by covering four key, interconnected areas:

1.  **Emergent Abilities**: We'll investigate why and how large models suddenly develop new skills that smaller models lack, a phenomenon known as emergence. We will visualize this "phase transition."
2.  **Intermediate-Guided Reasoning**: We will implement and analyze techniques like Chain-of-Thought (CoT) that unlock complex reasoning in LLMs by guiding them through intermediate steps. We'll also conceptually explore more advanced methods like Tree-of-Thought and Program-of-Thought.
3.  **The BabyLM Challenge**: We'll discuss the motivation behind training smaller, more efficient models on developmentally-plausible datasets, aiming to replicate human-like learning efficiency.
4.  **AI Agents**: The culmination of these concepts. We'll build a foundational agent framework from scratch, equipping an LLM with memory, tools, and a planning mechanism to act autonomously. We will also discuss multi-agent systems and the challenges of reliability, security, and alignment.

### 1.2 Prerequisites

To get the most out of this notebook, you should be familiar with the following concepts.

**Mathematical Concepts:**
- **Linear Algebra**: Vector spaces, dot products, and matrix operations (core to embeddings).
- **Probability & Statistics**: Basic concepts of probability distributions.
- **Information Theory**: High-level understanding of entropy and information.

**Machine Learning & Computer Science Concepts:**
- **Transformer Architecture**: Self-attention mechanism, encoders/decoders.
- **Large Language Models (LLMs)**: What they are, how they are pre-trained and fine-tuned.
- **Prompt Engineering**: Zero-shot, one-shot, and few-shot prompting.
- **APIs**: Basic understanding of how to interact with external services.
- **Basic Computer Architecture**: The roles of a CPU and RAM (useful for the LLM-as-OS analogy).

### 1.3 Learning Objectives

- **Understand and Visualize Emergence**: Simulate and plot the "phase transition" where abilities suddenly appear with scale.
- **Implement Chain-of-Thought**: Write code to contrast standard few-shot prompting with CoT and see the performance difference.
- **Build a Basic AI Agent**: Implement a simple agent loop (like ReAct) that gives an LLM access to tools and memory.
- **Grasp the Agent Ecosystem**: Understand the core components of an agent (planning, memory, tools) and the challenges in building robust, multi-agent systems.

**Estimated Time**: 2-3 hours.

## 2. The Phenomenon of Emergent Abilities

An ability is **emergent** if it is not present in smaller models but appears in larger models. This emergence is often a *phase transition*—performance is near-random until a certain model scale (a critical threshold), after which it increases sharply. This cannot be predicted by simply extrapolating the performance of smaller models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def plot_scaling_curves():
    """Visualize the difference between predictable scaling and emergent abilities."""
    
    # Model scale (e.g., in billions of parameters or training FLOPs)
    log_scale = np.linspace(0, 5, 50)
    
    # --- Predictable Scaling Law ---
    # Performance improves predictably with scale (e.g., loss decreases)
    # We model accuracy, so it increases.
    predictable_perf = 100 * (1 - np.exp(-0.8 * log_scale))
    # Add some noise
    predictable_perf += np.random.normal(0, 2, predictable_perf.shape)
    predictable_perf = np.clip(predictable_perf, 5, 95) # Clip to realistic values
    
    # --- Emergent Ability ---
    # Performance is near-random until a critical threshold, then jumps.
    random_chance = 10.0
    threshold = 3.0
    steepness = 5.0
    emergent_perf = random_chance + (95 - random_chance) / (1 + np.exp(-steepness * (log_scale - threshold)))
    emergent_perf += np.random.normal(0, 2.5, emergent_perf.shape)
    emergent_perf = np.clip(emergent_perf, 5, 95)

    # Create the plot
    plt.figure(figsize=(14, 6))

    # Plot for Predictable Scaling
    plt.subplot(1, 2, 1)
    plt.plot(log_scale, predictable_perf, marker='o', linestyle='-', color='b', label='Observed Performance')
    plt.axhline(y=random_chance, color='gray', linestyle='--', label='Random Chance')
    plt.title('Predictable Scaling Law')
    plt.xlabel('Model Scale (Log)')
    plt.ylabel('Task Performance (%)')
    plt.ylim(0, 100)
    plt.legend()

    # Plot for Emergent Ability
    plt.subplot(1, 2, 2)
    plt.plot(log_scale, emergent_perf, marker='o', linestyle='-', color='r', label='Observed Performance')
    plt.axvline(x=threshold, color='g', linestyle='--', label='Critical Threshold')
    plt.axhline(y=random_chance, color='gray', linestyle='--', label='Random Chance')
    plt.title('Emergent Ability (Phase Transition)')
    plt.xlabel('Model Scale (Log)')
    plt.ylabel('Task Performance (%)')
    plt.ylim(0, 100)
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_scaling_curves()

The plot on the left shows a task where performance improves smoothly and predictably. You could extrapolate the curve for smaller models and have a good guess about the performance of larger ones. 

The plot on the right demonstrates an **emergent ability**. Performance stays flat at random chance for a long time. Then, after crossing a "critical threshold" of scale, it shoots up dramatically. This sudden jump is the hallmark of emergence and is why these abilities are so surprising.

## 3. Intermediate-Guided Reasoning

One of the most significant emergent abilities is complex reasoning. Standard prompting often fails on multi-step problems because it asks the model to jump directly to the answer. **Intermediate-Guided Reasoning** techniques, like **Chain-of-Thought (CoT)**, unlock this capability by prompting the model to "think step-by-step."

In [ ]:
# We'll use a smaller, accessible model from Hugging Face for demonstration.
# Note: True CoT emergence is seen in much larger models (~100B+ params).
# With smaller models, CoT can still guide them to a better structure, but the effect is less dramatic.

import transformers
import torch

model_id = "gpt2"
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

def query_model(prompt, max_new_tokens=50):
    """Helper function to query the LLM and get the response."""
    outputs = pipeline(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False, # Use greedy decoding for reproducibility
        pad_token_id=pipeline.tokenizer.eos_token_id,
        num_return_sequences=1,
    )
    # Extract only the generated text, not the prompt
    full_text = outputs[0]['generated_text']
    generated_text = full_text[len(prompt):].strip()
    return generated_text

### 3.1 Standard Few-Shot Prompting vs. Chain-of-Thought

Let's test this on a simple multi-step arithmetic problem, as discussed in the lecture.

In [ ]:
def educational_standard_prompting():
    """Demonstrates standard few-shot prompting where the model gives the answer directly."""
    print("--- Testing Standard Few-Shot Prompting ---")
    
    # A simple example for the model to learn the format
    few_shot_example = """
Q: A farmer has 10 apples. He sells 3 and then buys 5 more. How many apples does he have?
A: 12
"""
    
    # The actual question we want to answer
    question = "Q: The cafeteria had 23 apples. They used 20 to make lunch and bought 6 more. How many apples do they have?"
    
    prompt = f"{few_shot_example}\n{question}\nA:"
    
    print(f"\nPROMPT:\n{prompt}")
    
    response = query_model(prompt, max_new_tokens=10)
    print(f"\nMODEL RESPONSE: {response}")
    print("Correct Answer: 9")

def educational_chain_of_thought_prompting():
    """
    Demonstrates CoT prompting where the model is shown how to reason step-by-step.
    """
    print("\n--- Testing Chain-of-Thought Prompting ---")

    # A CoT example showing the reasoning process
    cot_example = """
Q: A farmer has 10 apples. He sells 3 and then buys 5 more. How many apples does he have?
A: The farmer starts with 10 apples. He sells 3, so he has 10 - 3 = 7 apples. He then buys 5 more, so he has 7 + 5 = 12 apples. The final answer is 12.
"""

    # The actual question
    question = "Q: The cafeteria had 23 apples. They used 20 to make lunch and bought 6 more. How many apples do they have?"

    prompt = f"{cot_example}\n{question}\nA:"

    print(f"\nPROMPT:\n{prompt}")
    
    response = query_model(prompt, max_new_tokens=60)
    print(f"\nMODEL RESPONSE: {response}")
    print("Correct Answer: The cafeteria had 23 apples. They used 20, so they had 23 - 20 = 3. They bought 6 more, so they have 3 + 6 = 9. The final answer is 9.")


educational_standard_prompting()
educational_chain_of_thought_prompting()

**Observation**: Even with a small model like GPT-2, the CoT prompt provides a much better structure for the model to follow. While it might not get the arithmetic right, it attempts to follow the reasoning steps. Larger models would leverage this structure to arrive at the correct answer much more reliably. Standard prompting gives the smaller model almost no chance to solve the problem correctly.

### 3.2 Conceptual Implementations of Advanced Reasoning

Full implementations of methods like Tree-of-Thought are complex. Below, we provide conceptual, educational code structures to illustrate their core logic.

#### Program-of-Thought (PoT)

PoT offloads the reasoning to a deterministic code interpreter. The LLM's job is to translate the natural language problem into runnable code.

In [ ]:
def educational_program_of_thought():
    """
    Educational implementation of PoT.
    The LLM generates Python code, which we then execute.
    """
    print("--- Testing Program-of-Thought (Conceptual) ---")
    
    # Example showing how to convert a word problem to Python code
    pot_example = """
Q: A farmer has 10 apples. He sells 3 and then buys 5 more. How many apples does he have?
A: 
```python
apples = 10
apples -= 3
apples += 5
print(apples)
```
"""
    
    question = "Q: The cafeteria had 23 apples. They used 20 to make lunch and bought 6 more. How many apples do they have?"
    
    prompt = f"{pot_example}\n{question}\nA:"
    
    print(f"\nPROMPT:\n{prompt}")
    
    # For this, a code-generation model like CodeLlama or GPT-4 would be much better.
    # We'll simulate the ideal output for educational purposes.
    simulated_code_output = """
```python
cafeteria_apples = 23
cafeteria_apples -= 20
cafeteria_apples += 6
print(cafeteria_apples)
```
"""
    print(f"\nSIMULATED MODEL RESPONSE:\n{simulated_code_output}")
    
    # --- Execution Step ---
    try:
        code_to_execute = simulated_code_output.strip().split("```python").split("```")[0].strip()
        
        # Use a safe execution environment in a real application!
        # For this notebook, we'll use a dictionary to capture the print output.
        from io import StringIO
        import sys
        
        old_stdout = sys.stdout
        redirected_output = sys.stdout = StringIO()
        exec(code_to_execute)
        sys.stdout = old_stdout

        result = redirected_output.getvalue().strip()
        print(f"\nEXECUTED RESULT: {result}")
        print("Correct Answer: 9")
    except Exception as e:
        print(f"Error executing code: {e}")

educational_program_of_thought()

## 4. The Push for Efficiency: BabyLM

While scaling up models unlocks new abilities, it is incredibly resource-intensive. The **BabyLM Challenge** explores the other direction: can we train smaller models more efficiently by using higher-quality, developmentally-plausible data?

The key idea is that a child learns language from far less data than an LLM (e.g., ~100 million words by age 13 vs. trillions for models like Chinchilla). This data is also different—it's mostly transcribed speech and child-directed stories.


In [ ]:
!pip install datasets -q

from datasets import load_dataset

def explore_babylm_data():
    """Load and inspect a sample of the BabyLM dataset."""
    print("--- Exploring the BabyLM Dataset ---")
    try:
        # Load a small subset of the dataset for demonstration
        # We'll use the 'strict-small' configuration which is ~10M words
        dataset = load_dataset("cpllab/babylm_10M", split='train', streaming=True)
        
        print("\nSample entries from the BabyLM dataset:")
        count = 0
        for example in dataset:
            if count >= 5:
                break
            # The text is often messy, representing transcribed speech
            print(f"- {example['text'][:200]}...")
            count += 1
            
    except Exception as e:
        print(f"Could not load dataset. This may be due to network issues or dataset availability. Error: {e}")

explore_babylm_data()

**Motivation**: By studying how to pre-train effectively on smaller, higher-quality datasets, we can:
- Improve the efficiency of training all language models, large and small.
- Make state-of-the-art research more accessible to those without massive compute resources.
- Gain insights into human language acquisition.

## 5. AI Agents: From Language Models to Actors

An AI agent is a system that uses an LLM as its core "brain" or "CPU" to reason, plan, and **act** in an environment. A single call to an LLM is not enough; an agent requires a framework for **memory**, **tools**, and **planning**.

Let's build a simple, educational agent framework from scratch.

### 5.1 Step 1: Memory (A Simple Vector Store)
Long-term memory allows an agent to persist information across sessions. A common approach is to use a vector database to store and retrieve information based on semantic similarity. We'll implement a simplified, in-memory version.

First, we need a model to create embeddings (numerical representations) of text.

In [ ]:
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

class SimpleVectorMemory:
    """A simple in-memory vector store for an agent's memory."""
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.memory_vectors = []
        self.memory_texts = []

    def add_memory(self, text):
        """Add a piece of text to the memory."""
        if text in self.memory_texts:
            return
            
        print(f"🧠 ADDING MEMORY: '{text}'")
        embedding = self.embedding_model.encode(text, convert_to_tensor=True)
        self.memory_vectors.append(embedding)
        self.memory_texts.append(text)

    def retrieve_memory(self, query, top_k=1):
        """Retrieve the most relevant memories for a given query."""
        if not self.memory_vectors:
            return []
        
        query_embedding = self.embedding_model.encode(query, convert_to_tensor=True)
        
        # Stack memory vectors into a single tensor for efficient computation
        memory_tensor = torch.stack(self.memory_vectors)
        
        # Compute cosine similarity
        cos_scores = F.cosine_similarity(query_embedding, memory_tensor)
        
        # Get top_k results
        top_results = torch.topk(cos_scores, k=min(top_k, len(self.memory_vectors)))
        
        retrieved = [self.memory_texts[i] for i in top_results.indices]
        print(f"🔍 RETRIEVING MEMORY for '{query}': {retrieved}")
        return retrieved

### 5.2 Step 2: Tools
Tools are external functions the agent can call to get information or perform actions that the LLM cannot do on its own (e.g., perform accurate calculations, search the web, access a calendar).

In [ ]:
import math

class ToolBox:
    """A collection of tools the agent can use."""
    def calculate(self, expression: str) -> str:
        """Safe calculator for the agent."""
        try:
            # WARNING: In a real system, use a much safer eval method!
            allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith("__")}
            allowed_names["abs"] = abs
            result = str(eval(expression, {"__builtins__": {}}, allowed_names))
            return f"Result of '{expression}' is {result}"
        except Exception as e:
            return f"Error calculating '{expression}': {e}"
        
    def get_tools_description(self):
        return """
Available Tools:
- calculate(expression: str): Use this to perform any mathematical calculation. Example: calculate( (2+3)*5 )
"""

### 5.3 Step 3: The Agent (Planning and Execution)
The agent's core loop involves planning (deciding what to do next) and executing (using a tool or generating a final response). We'll implement a simple version of the **ReAct (Reason + Act)** framework.

The agent is prompted to think in a specific format:
1.  **Thought**: The agent reasons about the problem and decides what to do next.
2.  **Action**: The agent decides to use a tool. This will be parsed and executed.
3.  **Observation**: The result from the tool is fed back into the agent's context.
4.  ... The loop repeats until the agent has a final answer.

In [ ]:
import re

class SimpleReActAgent:
    """A simple agent that uses the ReAct framework."""
    def __init__(self, llm_pipeline, tools, memory):
        self.llm = llm_pipeline
        self.toolbox = tools
        self.memory = memory

    def educational_run(self, objective: str, max_steps=5):
        """Runs the agent loop with extensive comments for educational purposes."""
        
        # 1. Start with the base prompt including the objective, tool descriptions, and format instructions.
        retrieved_memories = self.memory.retrieve_memory(objective, top_k=2)
        memory_str = "\nRelevant Memories:\n" + "\n".join([f"- {mem}" for mem in retrieved_memories]) if retrieved_memories else ""
        
        prompt = f"""
You are a helpful assistant. Your objective is to: {objective}
{memory_str}
You must respond using the following format:
Thought: Your reasoning and plan to solve the objective.
Action: The tool to use, in the format `ToolName(argument)`. For example, `calculate(2*3)`.
If you have the final answer, respond ONLY with 'Final Answer: [your answer]'.

{self.toolbox.get_tools_description()}
Let's begin.
Objective: {objective}
"""

        history = ""

        for i in range(max_steps):
            print(f"\n--- STEP {i+1} ---")
            
            # 2. Construct the full prompt with the history of previous steps.
            full_prompt = prompt + history
            print(f"\nSENDING TO LLM:\n---\n{full_prompt}\n---")
            
            # 3. Query the LLM to get the next thought and action.
            response = query_model(full_prompt, max_new_tokens=100)
            print(f"LLM RESPONSE: {response}")
            
            history += response

            # 4. Check if the agent has provided a final answer.
            if "Final Answer:" in response:
                final_answer = response.split("Final Answer:")[-1].strip()
                print(f"\n✅ AGENT FINISHED: {final_answer}")
                # Add the successful interaction to memory
                self.memory.add_memory(f"Objective '{objective}' was successfully solved. Answer: {final_answer}")
                return final_answer

            # 5. Parse the action from the LLM's response.
            action_match = re.search(r"Action: (\w+)\((.*)\)", response)
            if action_match:
                tool_name = action_match.group(1).strip()
                tool_input = action_match.group(2).strip()
                
                # 6. Execute the action using the toolbox.
                print(f"🛠️ EXECUTING TOOL: {tool_name} with input '{tool_input}'")
                if hasattr(self.toolbox, tool_name):
                    tool_function = getattr(self.toolbox, tool_name)
                    observation = tool_function(tool_input)
                else:
                    observation = f"Error: Tool '{tool_name}' not found."
                
                print(f"👀 OBSERVATION: {observation}")
                history += f"\nObservation: {observation}\n"
            else:
                history += "\nObservation: Error - No valid action found. Please think again and provide an action."

        print("\n❌ AGENT STOPPED: Max steps reached.")
        return "Failed to reach a final answer."


### 5.4 Running the Agent

Now, let's instantiate all the components and run the agent on a task.

In [ ]:
# Instantiate the components
memory = SimpleVectorMemory(embedding_model)
toolbox = ToolBox()
agent = SimpleReActAgent(pipeline, toolbox, memory)

# Give the agent some prior knowledge
memory.add_memory("User is a baker.")
memory.add_memory("A baker's dozen is 13.")

# Run the agent on a task
objective = "If a baker makes 5 batches of cookies, and each batch is a baker's dozen, how many cookies did they make in total?"
agent.educational_run(objective)

**Observation**: This simple framework demonstrates the core loop of modern AI agents. The LLM acts as a reasoning engine to bridge the gap between a natural language objective and the concrete steps (tool use) needed to solve it. The memory provides context that the LLM might not have, leading to more personalized and accurate results.

## 6. Research Context & Extensions

The lecture highlighted several key research directions and challenges that build upon these ideas.

### 6.1 Multi-Agent Systems

Just as a single computer has limitations, a single agent does too. **Multi-agent systems** use a collection of specialized agents that can communicate and collaborate. This is often structured hierarchically, with a **manager** agent decomposing tasks and delegating them to **worker** agents.

In [ ]:
class ManagerAgent:
    """Conceptual model of a manager agent that delegates tasks."""
    def __init__(self, workers):
        self.workers = workers # A dictionary of worker agents, e.g., {'browser': BrowserAgent, ...}
        
    def pseudo_code_delegate(self, complex_objective):
        print(f"--- Manager received complex objective: '{complex_objective}' ---")
        
        # Step 1: Decompose the objective into sub-tasks (using an LLM call)
        sub_tasks = [
            {'task': 'Find the current price of stock XYZ', 'worker': 'browser'},
            {'task': 'Calculate 15% of that price', 'worker': 'calculator'}
        ]
        print(f"Decomposed into sub-tasks: {sub_tasks}")

        # Step 2: Delegate each sub-task to the appropriate worker
        results = {}
        for task_info in sub_tasks:
            worker = self.workers.get(task_info['worker'])
            if worker:
                print(f"Delegating '{task_info['task']}' to {task_info['worker']} worker.")
                # In a real system, this would be an async call
                # results[task_info['task']] = worker.run(task_info['task'])
            else:
                print(f"No worker found for task type: {task_info['worker']}")
                
        # Step 3: Aggregate results and form a final answer (another LLM call)
        print("Aggregating results to form the final answer...")
        
# This is purely for illustrating the concept
manager = ManagerAgent(workers={'browser': 'BrowserAgent_placeholder', 'calculator': 'CalculatorAgent_placeholder'})
manager.pseudo_code_delegate("What is 15% of the current price of stock XYZ?")

### 6.2 Challenges and Future Directions

This field is moving incredibly fast, but major challenges remain:

- **Reliability & The Looping Problem**: Agents can get stuck in loops or diverge from the objective. Developing robust error correction and feedback mechanisms is crucial.
- **Personalization & Alignment**: How do we align an agent with a user's specific preferences and values? This involves techniques like on-the-fly fine-tuning (e.g., LoRA) and learning from implicit/explicit feedback.
- **Security & Sandboxing**: Giving an agent control over a computer is risky. Research into sandboxing, permission models, and preventing prompt injection is critical for safe deployment.
- **The LLM OS**: The analogy of the LLM as a CPU in a new kind of operating system is powerful. Future research will likely involve building more of this "OS," including standardized protocols for memory access, tool use, and inter-agent communication.

## 7. Conclusion

This notebook has journeyed from the foundational properties of large-scale models to the complex systems being built on top of them. We've seen that:

1.  **Scale** is not just about doing the same things better; it unlocks qualitatively new **emergent abilities** like advanced reasoning.
2.  **Reasoning** can be explicitly guided through prompting techniques like **Chain-of-Thought**, bridging the gap between a problem and its solution.
3.  **Efficiency** is a growing concern, with initiatives like **BabyLM** exploring how to achieve more with less data, drawing inspiration from human learning.
4.  **Agents** are the next frontier, combining LLMs with memory, tools, and planning to create autonomous systems that can perform complex tasks in digital environments.

The shift from static models to dynamic, interactive agents represents a fundamental change in how we interact with and build AI, paving the way for the powerful, personalized assistants of the future.